In [ ]:
# this file looks at the local environment of each IF-annotated cell such as gene expression (predictors) and proportion dying (response).
# I used raw counts from the Imaris data and the previous file with spatial alignment of Visium bins and transcriptional data. 

# read in different dataframes depending on # of groups or individual genes of interest.

In [1]:
# importing necessary libraries
import pandas as pd
import numpy as np
from tqdm import tqdm # for the progress bar
import os

In [2]:
# setting my own working directory
os.chdir("i:/Hu Lab/Sophie/1. Cell death/visium image manual spot selection/20260413_final_merge/data")

In [3]:
# V = pd.read_csv("0511iso_sig.csv")                # visium, known signatures
# V = pd.read_feather("iso7spatial__genes.feather") # visium, all genes
# V = pd.read_csv("0511pd_sig.csv")
V = pd.read_csv("0513_NMF20.csv")

# S = pd.read_csv("iso7_coords_clean.csv") # spots from IF
S = pd.read_csv("pd1-9_coords_final.csv")  # spots from IF

In [4]:
gene_cols = V.columns[4:25]  # look at exact indices for category columns (not x,y).
# print(gene_cols)

In [ ]:
# parameters/ initializing things

In [5]:
v_coords = V[["x", "y"]].to_numpy()
s_coords = S[["x", "y"]].dropna().to_numpy()

In [ ]:
radius = 40 # what I ended up choosing to balance capturing enough cells but avoiding violating independence too much. 
            # Ideal was 10 microns but too sparse. each cell ~ 8 microns exist overlap & study simplifies to 2D ignoring z-axis.
            # if we use "windows" then can do overall expression in those and independence is preserved
            # attempt poisson sampling later. I hate bootstrapping

In [7]:
from scipy.spatial import cKDTree

In [ ]:
# Running the KNN tree to find neighbors

In [8]:
def inputs(S, V, gene_cols, s_coords, v_coords):

    # KD-trees
    s_tree = cKDTree(s_coords)
    v_tree = cKDTree(v_coords)

    # Cell types as integers --> faster processing
    type_map = {
        "tdtomato": 0,
        "gc3ai": 1,
        "cd8": 2,
        "lectin": 3
    }
    S_cells = np.array([type_map.get(x, -1) for x in S["cell_type"].values])

    # extracting gene matrix :)
    V_genes = V[gene_cols].to_numpy()

    return s_tree, v_tree, S_cells, V_genes

In [9]:
def counts_in_radius(center, s_tree, S_cells, radius):

    idx = s_tree.query_ball_point(center, r=radius)

    if len(idx) == 0:
        counts = np.zeros(4)   # for all four types, if nothing then set 0. Loops through all neighborhoods
    else:
        types = S_cells[idx]
        counts = np.bincount(types[types >= 0], minlength=4) # so our result: counts = [n_tdtomato, n_gc3ai, n_cd8, n_lectin]

    n_alive, n_dying, n_immune, n_endothelial = counts       # renaming the channels to what cell type they represent

    # calculating later metrics so I don't have to do it downstream
    tumor = n_alive + n_dying
    total = tumor + n_immune + n_endothelial
    prop_dying = n_dying / tumor if tumor > 0 else np.nan
    exist_dying = 1 if prop_dying > 0 else 0                        # binarizing proportion dying
    efficacy = prop_dying/ n_immune if n_immune > 0 else np.nan     # roportion dying per cd8, are some T-cells better at killing?

    return counts, tumor, total, prop_dying, exist_dying, efficacy

In [10]:
def get_gene_means(center, v_tree, V_genes, radius): # means have greater smoothing & 
                                                     # in our idea better representation of gene expression than sums.

    idx = v_tree.query_ball_point(center, r=radius)

    if len(idx) == 0:
        return np.zeros(V_genes.shape[1])

    return V_genes[idx].mean(axis=0)

In [11]:
def append_row(center, s_tree, v_tree, S_types, V_genes, radius):

    counts, tumor, total, prop_dying, exist_dying, efficacy = counts_in_radius(
        center, s_tree, S_types, radius
    )

    gene_means = get_gene_means(
        center, v_tree, V_genes, radius
    )

    row = np.concatenate([
        np.array([center[0], center[1]]),
        counts,
        np.array([tumor, total, prop_dying, exist_dying, efficacy]),
        gene_means
    ])

    return row

In [12]:
# final assembly. vectorizing is much better but less intuitive. 
# since it didn't take very long to run, I decided to prioritize readability.
def compute_neighborhoods(
    S, V, s_coords, v_coords, gene_cols, radius):

    s_tree, v_tree, S_types, V_genes = inputs(
        S, V, gene_cols, s_coords, v_coords
    )

    n_centers = len(s_coords)
    n_genes = V_genes.shape[1]

    results = np.zeros((n_centers, 11 + n_genes))

    for i, center in enumerate(tqdm(s_coords, desc="Processing")):
        results[i] = append_row(
            center, s_tree, v_tree, S_types, V_genes, radius
        )

    columns = (
        ["cx", "cy",
         "n_alive", "n_dying", "n_immune", "n_lectin",
         "tumor", "all", "prop_dying", "exist_dying", "efficacy"]
        + list(gene_cols)
    )

    return pd.DataFrame(results, columns=columns)

In [13]:
# actually running the function now. depending on how many genes or signatures, after the progress bar ends, 
# it may still take a while. just a heads up, sorry

df = compute_neighborhoods(
    S=S,
    V=V,
    s_coords=s_coords,
    v_coords=v_coords,
    gene_cols=gene_cols,
    radius=radius
)

Processing: 100%|██████████| 88019/88019 [00:03<00:00, 29064.40it/s]


In [14]:
# cleaning a bit. Set tumor n = 30 because of CLT. 
# neighborhoods overlap spatially so not fully independent but capturing too few cells --> high variance

df = df[df["tumor"] > 30]
df = df[df["n_immune"] > 0]                # removing for NaN efficacy denominator
df = df.join(S[["cell_type", "sample"]])   # maintaining cell type and sample info for downstream

In [ ]:
# seeing max possible number of neighborhoods given the area and radius.
area_estimate = (df['cx'].max()-df['cx'].min()) * (df['cy'].max()-df['cy'].min())
max_possible = area_estimate / (np.pi * radius**2)
print(max_possible)

4639.801863443076


In [ ]:
# sanity checks. 
print(df[['cx','cy']].describe())
print(df[['cx','cy']].isna().sum())
print(df[['cx','cy']].isna().sum())

                cx           cy
count  8853.000000  8853.000000
mean   4275.967215  4893.470673
std    1075.010383  1267.320205
min    1167.213000  1739.674000
25%    3783.146000  4119.446000
50%    4406.913000  5386.864000
75%    4971.367000  5897.014000
max    6224.935000  6350.878000
cx    0
cy    0
dtype: int64
cx    0
cy    0
dtype: int64


In [30]:
def sample_non_overlapping_simple(df, radius, seed):
    rng = np.random.default_rng(seed)

    coords = df[['cx', 'cy']].to_numpy()
    remaining_idx = np.arange(len(coords))

    selected_idx = []

    while len(remaining_idx) > 0:
        # pick a random remaining point
        pick_i = rng.choice(remaining_idx)
        selected_idx.append(pick_i)

        # build tree of remaining points
        tree = cKDTree(coords[remaining_idx])

        # find all points within radius of the chosen point
        neighbors = tree.query_ball_point(coords[pick_i], r=radius)

        # map neighbor indices back to global indices
        to_remove = set(remaining_idx[neighbors])

        # keep only points not removed
        remaining_idx = np.array([i for i in remaining_idx if i not in to_remove])

    return df.iloc[selected_idx].copy()

In [31]:
df_sub = sample_non_overlapping_simple(df, 
                                       radius=radius, seed=42) # hitchiker's guide to the galaxy! the answer to the ultimate question of life, the universe, and everything.

In [32]:
# verify output
print(df_sub.iloc[400:405, ])
df_sub.shape

             cx        cy  n_alive  n_dying  n_immune  n_lectin  tumor   all  \
87731  4669.102  6010.967     32.0      0.0       7.0       2.0   32.0  41.0   
71442  4204.621  6044.421     35.0      1.0       2.0       0.0   36.0  38.0   
46301  5726.112  4413.792     29.0      2.0       2.0       0.0   31.0  33.0   
82335  2192.077  3209.058     27.0      5.0       3.0       4.0   32.0  39.0   
61205  3381.971  5317.744     31.0      0.0       1.0       0.0   31.0  32.0   

       prop_dying  exist_dying  ...        13        14            15  \
87731    0.000000          0.0  ...  0.000002  0.000006  3.356769e-06   
71442    0.027778          1.0  ...  0.000008  0.000005  1.018977e-06   
46301    0.064516          1.0  ...  0.000003  0.000004  2.536559e-06   
82335    0.156250          1.0  ...  0.000005  0.000006  1.333901e-06   
61205    0.000000          0.0  ...  0.000003  0.000003  5.539315e-07   

             16        17            18        19            20  cell_type  \
87

(822, 33)

In [ ]:
# df.to_feather("spatial_means_all.feather")    # feather is for all genes, too large a file
df.to_csv("0514_NMF20calc.csv", index=False) 